# Using Expressions (polars only)

This notebook demonstrates the `mintalib.expressions` module — polars expression factory functions for use with `select` and `with_columns`.

Expressions are named in **upper case** (e.g. `SMA`, `EMA`, `MACD`). An optional `src` keyword parameter sets the source expression; by default series-based indicators use `pl.col("close")`. Multi-column outputs like `MACD` return a polars struct expression — use `.struct.unnest()` to flatten into separate columns, or `.struct.field(name)` to pick one. Expressions can be chained using `.pipe()`.

In [1]:
import polars as pl
from mintalib.expressions import SMA, EMA, ATR, MACD, ROC

In [2]:
from mintalib.samples import sample_prices

prices = sample_prices(backend="polars")
prices


date,open,high,low,close,volume
date,f64,f64,f64,f64,i64
1980-12-12,0.098298,0.098725,0.098298,0.098298,469033600
1980-12-15,0.093597,0.093597,0.093169,0.093169,175884800
1980-12-16,0.086758,0.086758,0.086331,0.086331,105728000
1980-12-17,0.088468,0.088895,0.088468,0.088468,86441600
1980-12-18,0.091032,0.09146,0.091032,0.091032,73449600
…,…,…,…,…,…
2026-04-20,270.329987,274.279999,270.290009,273.049988,36590200
2026-04-21,271.5,272.799988,265.399994,266.170013,50209800
2026-04-22,267.820007,273.73999,266.869995,273.170013,43249200


In [3]:
prices.select(
    SMA(20).alias("ema20")
)


ema20
f64
null
null
null
null
null
…
257.638998
258.372999
259.4495


In [4]:
prices.select(
    ATR(14).alias("atr")
)



atr
f64
null
null
null
null
null
…
5.964207
6.084621
6.190717


In [5]:
prices.select(
    MACD().struct.unnest()
)

macd,macdsignal,macdhist
f64,f64,f64
null,null,null
null,null,null
null,null,null
null,null,null
null,null,null
…,…,…
2.872127,0.764249,2.107878
2.926444,1.196688,1.729756
3.494055,1.656161,1.837893


In [6]:
# Expression can be piped into other expression factory functions.
# The first expression argument is passed as `src` to the next function.

prices.select(
    EMA(20).pipe(ROC, 1)
)

roc
f64
null
null
null
null
null
…
0.005064
0.002044
0.004399


In [7]:
prices.select(
    MACD().struct.unnest(),
    sma=SMA(50),
    atr=ATR(14),
    trend=EMA(50).pipe(ROC, 1)
)

macd,macdsignal,macdhist,sma,atr,trend
f64,f64,f64,f64,f64,f64
null,null,null,null,null,null
null,null,null,null,null,null
null,null,null,null,null,null
null,null,null,null,null,null
null,null,null,null,null,null
…,…,…,…,…,…
2.872127,0.764249,2.107878,260.501999,5.964207,0.002049
2.926444,1.196688,1.729756,260.268199,6.084621,0.000927
3.494055,1.656161,1.837893,260.2392,6.190717,0.001944
